# Parte 1 — Base: dados EMNIST-36 + VAE
**Trabalho Final — IA Generativa (UFMG 2026/2)**

Este notebook:
1. carrega o EMNIST com as **36 classes** (`0`–`9`, `A`–`Z`);
2. treina o **VAE** ($\mu_E, \sigma_E, \mu_D$) com a loss $R(x) + \beta L(x)$ da Parte 5;
3. valida o VAE (reconstruções, amostras, espaço latente);
4. exporta **`vae.pt`** e **`latents.pt`**, que são a entrada do Flow ($\Phi$).

➡️ No Colab: *Ambiente de execução → Alterar tipo → GPU (T4)*. O treino completo leva poucos minutos.

In [ ]:
!nvidia-smi -L || echo 'Sem GPU: vai funcionar, mas bem mais devagar'

## 1. Código da base
As células abaixo gravam os módulos (`src/`) e o script de treino. São cópias exatas dos arquivos do repositório.

In [ ]:
!mkdir -p src outputs

In [ ]:
%%writefile src/__init__.py
"""
Trabalho Final de IA Generativa (UFMG 2026/2):
VAE + Rectified Flow no espaço latente, dataset EMNIST (36 classes).

Base do projeto (dados + VAE):
    src.dataset  -> carregamento do EMNIST com 36 classes ('0'-'9', 'A'-'Z')
    src.vae      -> VAE (mu_E, sigma_E, mu_D), loss da Parte 5, salvar/carregar
    src.vis_vae  -> figuras do VAE (reconstruções, amostras, espaço latente)
"""


In [ ]:
%%writefile src/dataset.py
"""
Carregamento do EMNIST com exatamente 36 classes: 10 dígitos ('0'-'9') + 26 letras ('A'-'Z').

Escolha do split
----------------
O EMNIST não tem um split pronto com 36 classes. Usamos o split 'balanced' (47 classes):
    rótulos  0..9   -> dígitos '0'..'9'
    rótulos 10..35  -> letras  'A'..'Z'  (maiúsculas; minúsculas parecidas, como c/C e o/O, já vêm juntas)
    rótulos 36..46  -> minúsculas distintas (a, b, d, e, f, g, h, n, q, r, t)  -> DESCARTADAS
Filtrando os rótulos < 36 ficamos com 36 classes balanceadas:
    treino: 2400 imagens/classe  -> 86.400 imagens
    teste :  400 imagens/classe  -> 14.400 imagens

Orientação
----------
As imagens do EMNIST no torchvision vêm transpostas (de lado). Corrigimos trocando H e W.

Formato de saída: tensores x em [0, 1] com shape (1, 28, 28) e rótulo y em {0, ..., 35}.
"""
import string
from typing import Tuple

import torch
from torch.utils.data import DataLoader, Dataset, Subset, TensorDataset

# Mapeamento das 36 classes: 0..9 -> '0'..'9', 10..35 -> 'A'..'Z'
CLASS_NAMES = list(string.digits) + list(string.ascii_uppercase)
NUM_CLASSES = 36
IMG_SHAPE = (1, 28, 28)


def index_to_char(idx: int) -> str:
    """Índice da classe (0..35) -> caractere ('0'-'9', 'A'-'Z')."""
    return CLASS_NAMES[idx] if 0 <= idx < NUM_CLASSES else "?"


def char_to_index(ch: str) -> int:
    """Caractere ('0'-'9' ou 'A'-'Z', sem diferenciar maiúscula/minúscula) -> índice 0..35."""
    ch = ch.upper()
    if ch in CLASS_NAMES:
        return CLASS_NAMES.index(ch)
    raise ValueError(f"Caractere '{ch}' inválido. Use '0'-'9' ou 'A'-'Z'.")


class TransposeEMNIST:
    """Corrige a orientação das imagens do EMNIST (troca altura e largura)."""

    def __call__(self, tensor: torch.Tensor) -> torch.Tensor:
        return tensor.transpose(-2, -1).contiguous()


def get_emnist_36_datasets(data_dir: str = "./data") -> Tuple[Dataset, Dataset]:
    """Retorna (treino, teste) do EMNIST com as 36 classes, imagens em [0, 1] e orientação corrigida."""
    from torchvision import datasets, transforms

    transform = transforms.Compose([transforms.ToTensor(), TransposeEMNIST()])

    train_full = datasets.EMNIST(root=data_dir, split="balanced", train=True, download=True, transform=transform)
    test_full = datasets.EMNIST(root=data_dir, split="balanced", train=False, download=True, transform=transform)

    train_idx = torch.where(train_full.targets < NUM_CLASSES)[0].tolist()
    test_idx = torch.where(test_full.targets < NUM_CLASSES)[0].tolist()
    return Subset(train_full, train_idx), Subset(test_full, test_idx)


def get_fake_datasets(n_train: int = 2048, n_test: int = 512, seed: int = 0) -> Tuple[Dataset, Dataset]:
    """
    Dados falsos com o mesmo formato do EMNIST-36 (para testar o código sem baixar nada).
    Cada classe é um "borrão" numa posição diferente, para o VAE ter algo para aprender.
    """
    g = torch.Generator().manual_seed(seed)

    def make(n: int) -> TensorDataset:
        y = torch.randint(0, NUM_CLASSES, (n,), generator=g)
        yy, xx = torch.meshgrid(torch.arange(28.0), torch.arange(28.0), indexing="ij")
        cy = 6 + (y // 6).float() * 3.2
        cx = 6 + (y % 6).float() * 3.2
        d2 = (yy[None] - cy[:, None, None]) ** 2 + (xx[None] - cx[:, None, None]) ** 2
        x = torch.exp(-d2 / 18.0) + 0.05 * torch.rand(n, 28, 28, generator=g)
        return TensorDataset(x.clamp(0, 1).unsqueeze(1), y)

    return make(n_train), make(n_test)


def get_dataloaders(
    data_dir: str = "./data",
    batch_size: int = 256,
    num_workers: int = 2,
    fake: bool = False,
) -> Tuple[DataLoader, DataLoader]:
    """DataLoaders de treino (embaralhado) e teste. Use fake=True para um teste rápido sem internet."""
    train_ds, test_ds = get_fake_datasets() if fake else get_emnist_36_datasets(data_dir)
    pin = torch.cuda.is_available()
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=pin, drop_last=True
    )
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=pin)
    return train_loader, test_loader


In [ ]:
%%writefile src/vae.py
"""
VAE da Parte 5: encoder probabilístico (mu_E, sigma_E) e decoder mu_D.

    Encoder:  Z(x) = mu_E(x) + sigma_E(x) * W_E,   W_E ~ N(0, I)
    Decoder:  x_hat = mu_D(z)

    Objetivo (slides 55-59):  min  sum_i  R(x_i) + beta * L(x_i)
        R(x) = E_W [ ||x - mu_D(mu_E(x) + sigma_E(x) W)||^2 / 2 ]
        L(x) = m * ((sigma_E^2 - 1)/2 - ln sigma_E) + ||mu_E(x)||^2 / 2        (sigma_E escalar)
             = sum_j ((sigma_j^2 - 1)/2 - ln sigma_j) + ||mu_E(x)||^2 / 2    (sigma_E diagonal)

    beta = 1 é exatamente a fórmula do enunciado. Com R(x) = ||x - x_hat||^2 / 2 em pixels [0, 1], o termo R
    fica pequeno (~10) perto de L, e beta = 1 tende a "desligar" dimensões latentes e borrar as imagens.
    beta < 1 (padrão do treino: 0.25) dá reconstruções mais nítidas; o Flow corrige depois a distribuição
    do latente, então não precisamos de um latente perfeitamente N(0, I).

Interface usada pelo resto do grupo:
    vae = load_vae("outputs/vae.pt", device)
    mu, sigma = vae.encode(x)       # x: (B, 1, 28, 28) em [0, 1]
    x_hat = vae.decode(z)           # z: (B, m)
"""
from typing import Dict, Tuple

import torch
import torch.nn as nn
from torch.utils.data import DataLoader


class Encoder(nn.Module):
    """(B, 1, 28, 28) -> mu_E (B, m), log sigma_E (B, m) ou (B, 1)."""

    def __init__(self, latent_dim: int = 16, scalar_sigma: bool = False):
        super().__init__()
        # (1, 28, 28) -> (32, 14, 14) -> (64, 7, 7) -> (128, 4, 4)
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.SiLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.SiLU(),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.SiLU(),
            nn.Flatten(),
        )
        self.fc = nn.Sequential(nn.Linear(128 * 4 * 4, 256), nn.SiLU())
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_log_sigma = nn.Linear(256, 1 if scalar_sigma else latent_dim)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        h = self.fc(self.conv(x))
        mu = self.fc_mu(h)
        log_sigma = self.fc_log_sigma(h).clamp(-10.0, 5.0)  # estabilidade numérica
        return mu, log_sigma


class Decoder(nn.Module):
    """(B, m) -> imagem (B, 1, 28, 28) em [0, 1]."""

    def __init__(self, latent_dim: int = 16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.SiLU(),
            nn.Linear(256, 128 * 7 * 7), nn.SiLU(),
        )
        # (128, 7, 7) -> (64, 14, 14) -> (32, 28, 28) -> (1, 28, 28)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.BatchNorm2d(64), nn.SiLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1), nn.BatchNorm2d(32), nn.SiLU(),
            nn.Conv2d(32, 1, 3, padding=1), nn.Sigmoid(),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.deconv(self.fc(z).view(-1, 128, 7, 7))


class VAE(nn.Module):
    def __init__(self, latent_dim: int = 16, scalar_sigma: bool = False):
        super().__init__()
        self.latent_dim = latent_dim
        self.scalar_sigma = scalar_sigma
        self.encoder = Encoder(latent_dim, scalar_sigma)
        self.decoder = Decoder(latent_dim)

    @property
    def config(self) -> Dict:
        return {"latent_dim": self.latent_dim, "scalar_sigma": self.scalar_sigma}

    def encode(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Retorna (mu_E(x), sigma_E(x)). sigma_E tem shape (B, m) ou (B, 1)."""
        mu, log_sigma = self.encoder(x)
        return mu, log_sigma.exp()

    @staticmethod
    def reparameterize(mu: torch.Tensor, sigma: torch.Tensor) -> torch.Tensor:
        """Z(x) = mu_E(x) + sigma_E(x) * W,  W ~ N(0, I)."""
        return mu + sigma * torch.randn_like(mu)

    def decode(self, z: torch.Tensor) -> torch.Tensor:
        """x_hat = mu_D(z)."""
        return self.decoder(z)

    def forward(self, x: torch.Tensor):
        mu, sigma = self.encode(x)
        z = self.reparameterize(mu, sigma)
        return self.decode(z), mu, sigma, z

    def compute_loss(self, x: torch.Tensor, beta: float = 1.0, num_mc_samples: int = 1) -> Dict[str, torch.Tensor]:
        """R(x) + beta * L(x), média no batch. Ver docstring do módulo."""
        mu, log_sigma = self.encoder(x)
        sigma = log_sigma.exp()

        # R(x): Monte Carlo com K = num_mc_samples amostras de W
        recon = 0.0
        for _ in range(num_mc_samples):
            x_hat = self.decode(self.reparameterize(mu, sigma))
            recon = recon + 0.5 * ((x - x_hat) ** 2).sum(dim=(1, 2, 3))
        recon = (recon / num_mc_samples).mean()

        # L(x): fórmula fechada (KL entre N(mu, sigma^2 I) e N(0, I))
        per_dim = 0.5 * (sigma ** 2 - 1.0) - log_sigma
        if self.scalar_sigma:
            sigma_term = self.latent_dim * per_dim.squeeze(-1)
        else:
            sigma_term = per_dim.sum(dim=1)
        latent = (sigma_term + 0.5 * (mu ** 2).sum(dim=1)).mean()

        return {"loss": recon + beta * latent, "recon_error": recon.detach(), "latent_error": latent.detach()}


# --------------------------------------------------------------------------------------
# Salvar / carregar
# --------------------------------------------------------------------------------------
def save_vae(vae: VAE, path: str, **extra) -> None:
    torch.save({"config": vae.config, "state_dict": vae.state_dict(), **extra}, path)


def load_vae(path: str, device="cpu") -> VAE:
    ckpt = torch.load(path, map_location=device)
    vae = VAE(**ckpt["config"]).to(device)
    vae.load_state_dict(ckpt["state_dict"])
    vae.eval()
    return vae


@torch.no_grad()
def encode_dataset(vae: VAE, loader: DataLoader, device) -> Dict[str, torch.Tensor]:
    """Codifica um dataset inteiro: retorna {'mu': (N, m), 'sigma': (N, m|1), 'y': (N,)} na CPU."""
    vae.eval()
    mus, sigmas, ys = [], [], []
    for x, y in loader:
        mu, sigma = vae.encode(x.to(device))
        mus.append(mu.cpu())
        sigmas.append(sigma.cpu())
        ys.append(y.cpu())
    return {"mu": torch.cat(mus), "sigma": torch.cat(sigmas), "y": torch.cat(ys)}


In [ ]:
%%writefile src/vis_vae.py
"""Figuras do VAE (para validar a base e usar nos slides)."""
from typing import Dict, List, Optional

import matplotlib.pyplot as plt
import torch

from .dataset import CLASS_NAMES, NUM_CLASSES, index_to_char
from .vae import VAE


def _finish(fig, save_path: Optional[str], show: bool):
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
    if show:
        plt.show()
    else:
        plt.close(fig)


def plot_dataset_grid(images, labels, n: int = 36, save_path=None, show=True):
    """Grade com exemplos reais do dataset (confere orientação e rótulos)."""
    cols = 12
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.0, rows * 1.15))
    for i, ax in enumerate(axes.flat):
        ax.axis("off")
        if i < n:
            ax.imshow(images[i, 0].cpu(), cmap="gray", vmin=0, vmax=1)
            ax.set_title(index_to_char(int(labels[i])), fontsize=9)
    fig.suptitle("EMNIST-36: exemplos reais", fontsize=11)
    _finish(fig, save_path, show)


def plot_loss_curves(history: Dict[str, List[float]], save_path=None, show=True):
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.3))
    for ax, key, title in zip(
        axes, ["loss", "recon_error", "latent_error"], ["Loss total", "Reconstrução R(x)", "Latente L(x)"]
    ):
        ax.plot(history[f"train_{key}"], label="treino")
        ax.plot(history[f"test_{key}"], label="teste")
        ax.set_title(title)
        ax.set_xlabel("época")
        ax.legend()
        ax.grid(alpha=0.3)
    _finish(fig, save_path, show)


@torch.no_grad()
def plot_reconstructions(vae: VAE, images, labels=None, n: int = 12, save_path=None, show=True):
    """Topo: x original. Baixo: x_hat = mu_D(mu_E(x))."""
    vae.eval()
    device = next(vae.parameters()).device
    x = images[:n].to(device)
    mu, _ = vae.encode(x)
    x_hat = vae.decode(mu).cpu()
    fig, axes = plt.subplots(2, n, figsize=(n * 1.1, 2.6))
    for i in range(n):
        axes[0, i].imshow(x[i, 0].cpu(), cmap="gray", vmin=0, vmax=1)
        axes[1, i].imshow(x_hat[i, 0], cmap="gray", vmin=0, vmax=1)
        axes[0, i].axis("off")
        axes[1, i].axis("off")
        if labels is not None:
            axes[0, i].set_title(index_to_char(int(labels[i])), fontsize=9)
    fig.suptitle("Original (topo) vs. reconstrução μ_D(μ_E(x)) (baixo)", fontsize=11)
    _finish(fig, save_path, show)


@torch.no_grad()
def plot_prior_samples(vae: VAE, n: int = 64, seed: int = 0, save_path=None, show=True):
    """
    Decodifica z ~ N(0, I) direto, SEM o Flow. Serve de baseline:
    o Flow deve gerar caracteres melhores que isso.
    """
    vae.eval()
    device = next(vae.parameters()).device
    g = torch.Generator(device="cpu").manual_seed(seed)
    z = torch.randn(n, vae.latent_dim, generator=g).to(device)
    imgs = vae.decode(z).cpu()
    cols = 8
    fig, axes = plt.subplots(n // cols, cols, figsize=(cols * 1.0, n // cols * 1.0))
    for ax, img in zip(axes.flat, imgs):
        ax.imshow(img[0], cmap="gray", vmin=0, vmax=1)
        ax.axis("off")
    fig.suptitle("VAE sozinho: μ_D(z), z ~ N(0, I)  (baseline sem Flow)", fontsize=10)
    _finish(fig, save_path, show)


@torch.no_grad()
def plot_interpolation(vae: VAE, x_a, x_b, steps: int = 10, save_path=None, show=True):
    """Interpolação linear no latente entre duas imagens: mostra se o espaço latente é 'suave'."""
    vae.eval()
    device = next(vae.parameters()).device
    mu_a, _ = vae.encode(x_a[None].to(device))
    mu_b, _ = vae.encode(x_b[None].to(device))
    t = torch.linspace(0, 1, steps, device=device)[:, None]
    imgs = vae.decode((1 - t) * mu_a + t * mu_b).cpu()
    fig, axes = plt.subplots(1, steps, figsize=(steps * 1.1, 1.4))
    for ax, img in zip(axes, imgs):
        ax.imshow(img[0], cmap="gray", vmin=0, vmax=1)
        ax.axis("off")
    fig.suptitle("Interpolação linear no espaço latente", fontsize=10)
    _finish(fig, save_path, show)


def plot_latent_pca(mu: torch.Tensor, y: torch.Tensor, max_points: int = 8000, save_path=None, show=True):
    """Projeção 2D (PCA) dos mu_E(x), colorida por classe, com o caractere no centro de cada classe."""
    idx = torch.randperm(len(mu))[:max_points]
    m, yy = mu[idx], y[idx]
    m = m - m.mean(0)
    _, _, v = torch.pca_lowrank(m, q=2, center=False)
    p = m @ v[:, :2]
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(p[:, 0], p[:, 1], c=yy, cmap="nipy_spectral", s=2, alpha=0.5, vmin=0, vmax=NUM_CLASSES - 1)
    for c in range(NUM_CLASSES):
        sel = yy == c
        if sel.any():
            cx, cy = p[sel].mean(0)
            ax.text(float(cx), float(cy), CLASS_NAMES[c], fontsize=10, weight="bold", ha="center", va="center")
    ax.set_title("Espaço latente μ_E(x) — PCA 2D")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    _finish(fig, save_path, show)


In [ ]:
%%writefile train_vae.py
"""
Base do trabalho: treina o VAE no EMNIST-36 e exporta o que o resto do grupo precisa.

Uso (Colab com GPU):
    python train_vae.py                      # treino completo (padrão: m=16, beta=0.25, 30 épocas)
    python train_vae.py --beta 1.0           # fórmula exata do enunciado (latente mais gaussiano, imagens mais borradas)
    python train_vae.py --fake --epochs 2    # teste rápido sem baixar o EMNIST

Saídas em --out-dir (padrão ./outputs):
    vae.pt        -> pesos + config do VAE           (carregar com src.vae.load_vae)
    latents.pt    -> mu_E(x), sigma_E(x) e rótulos de treino/teste (entrada do Flow)
    metrics.json  -> hiperparâmetros e curvas de loss
    *.png         -> figuras de validação (reconstrução, amostras, PCA, interpolação, curvas)
"""
import argparse
import json
import os
import time

import matplotlib

matplotlib.use("Agg")
import torch  # noqa: E402

from src.dataset import CLASS_NAMES, get_dataloaders  # noqa: E402
from src.vae import VAE, encode_dataset, save_vae  # noqa: E402
from src import vis_vae  # noqa: E402


@torch.no_grad()
def evaluate(vae: VAE, loader, device, beta: float):
    vae.eval()
    tot = {"loss": 0.0, "recon_error": 0.0, "latent_error": 0.0}
    n = 0
    for x, _ in loader:
        x = x.to(device)
        out = vae.compute_loss(x, beta=beta)
        for k in tot:
            tot[k] += float(out[k]) * len(x)
        n += len(x)
    return {k: v / n for k, v in tot.items()}


def main():
    p = argparse.ArgumentParser(description="Treino do VAE no EMNIST (36 classes)")
    p.add_argument("--data-dir", default="./data")
    p.add_argument("--out-dir", default="./outputs")
    p.add_argument("--latent-dim", type=int, default=16, help="dimensão m do espaço latente")
    p.add_argument("--scalar-sigma", action="store_true", help="sigma_E escalar (fórmula com m* do enunciado)")
    p.add_argument("--beta", type=float, default=0.25,
                   help="peso de L(x). 1.0 = fórmula exata do enunciado; <1 = reconstruções mais nítidas")
    p.add_argument("--epochs", type=int, default=30)
    p.add_argument("--batch-size", type=int, default=256)
    p.add_argument("--lr", type=float, default=1e-3)
    p.add_argument("--num-workers", type=int, default=2)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--fake", action="store_true", help="usa dados falsos (teste do código sem internet)")
    args = p.parse_args()

    torch.manual_seed(args.seed)
    os.makedirs(args.out_dir, exist_ok=True)
    out = lambda name: os.path.join(args.out_dir, name)  # noqa: E731
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Dispositivo: {device}")

    # 1) Dados
    train_loader, test_loader = get_dataloaders(args.data_dir, args.batch_size, args.num_workers, fake=args.fake)
    print(f"Treino: {len(train_loader.dataset)} imagens | Teste: {len(test_loader.dataset)} imagens")
    x_vis, y_vis = next(iter(test_loader))
    vis_vae.plot_dataset_grid(x_vis, y_vis, save_path=out("dataset_grid.png"), show=False)

    # 2) Modelo
    vae = VAE(args.latent_dim, args.scalar_sigma).to(device)
    n_params = sum(p.numel() for p in vae.parameters())
    print(f"VAE: m={args.latent_dim}, sigma {'escalar' if args.scalar_sigma else 'diagonal'}, "
          f"beta={args.beta}, {n_params/1e6:.2f}M parâmetros")
    opt = torch.optim.AdamW(vae.parameters(), lr=args.lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.epochs)

    # 3) Treino
    history = {f"{s}_{k}": [] for s in ("train", "test") for k in ("loss", "recon_error", "latent_error")}
    best_test = float("inf")
    t0 = time.time()
    for epoch in range(1, args.epochs + 1):
        vae.train()
        tot = {"loss": 0.0, "recon_error": 0.0, "latent_error": 0.0}
        n = 0
        for x, _ in train_loader:
            x = x.to(device, non_blocking=True)
            losses = vae.compute_loss(x, beta=args.beta)
            opt.zero_grad(set_to_none=True)
            losses["loss"].backward()
            opt.step()
            for k in tot:
                tot[k] += losses[k].item() * len(x)
            n += len(x)
        sched.step()
        train_m = {k: v / n for k, v in tot.items()}
        test_m = evaluate(vae, test_loader, device, args.beta)
        for k in tot:
            history[f"train_{k}"].append(train_m[k])
            history[f"test_{k}"].append(test_m[k])
        print(f"[{epoch:02d}/{args.epochs}] treino: loss {train_m['loss']:.2f} "
              f"(R {train_m['recon_error']:.2f}, L {train_m['latent_error']:.2f}) | "
              f"teste: loss {test_m['loss']:.2f} (R {test_m['recon_error']:.2f}, L {test_m['latent_error']:.2f}) "
              f"| {time.time() - t0:.0f}s")
        if test_m["loss"] < best_test:
            best_test = test_m["loss"]
            save_vae(vae, out("vae.pt"), epoch=epoch, test_metrics=test_m, beta=args.beta)

    # Recarrega o melhor checkpoint (menor loss de teste)
    ckpt = torch.load(out("vae.pt"), map_location=device)
    vae.load_state_dict(ckpt["state_dict"])
    print(f"Melhor época: {ckpt['epoch']} (loss de teste {best_test:.2f}) -> {out('vae.pt')}")

    # 4) Exporta os latentes para o Flow
    # (loaders sem embaralhar para os índices baterem com o dataset)
    plain_train = torch.utils.data.DataLoader(train_loader.dataset, batch_size=512, shuffle=False,
                                              num_workers=args.num_workers)
    lat_train = encode_dataset(vae, plain_train, device)
    lat_test = encode_dataset(vae, test_loader, device)
    mu = lat_train["mu"]
    # dimensões "ativas": variância de mu_E(x) entre imagens > 0.01 (as outras o VAE não usa)
    active_dims = int((mu.var(0) > 0.01).sum())
    torch.save(
        {
            "train": lat_train,
            "test": lat_test,
            "latent_dim": args.latent_dim,
            "class_names": CLASS_NAMES,
            "mu_mean": mu.mean(0),  # estatísticas para normalizar o latente no Flow, se quiserem
            "mu_std": mu.std(0),
        },
        out("latents.pt"),
    )
    print(f"Latentes salvos em {out('latents.pt')}: treino {tuple(mu.shape)}, teste {tuple(lat_test['mu'].shape)}")
    print(f"Estatísticas de mu_E(x): média {mu.mean():.3f}, desvio {mu.std():.3f}, "
          f"dimensões ativas {active_dims}/{args.latent_dim}")

    # 5) Figuras
    vis_vae.plot_loss_curves(history, save_path=out("vae_loss_curves.png"), show=False)
    vis_vae.plot_reconstructions(vae, x_vis, y_vis, save_path=out("vae_reconstructions.png"), show=False)
    vis_vae.plot_prior_samples(vae, save_path=out("vae_prior_samples.png"), show=False)
    vis_vae.plot_interpolation(vae, x_vis[0], x_vis[1], save_path=out("vae_interpolation.png"), show=False)
    vis_vae.plot_latent_pca(lat_test["mu"], lat_test["y"], save_path=out("vae_latent_pca.png"), show=False)

    with open(out("metrics.json"), "w") as f:
        json.dump({"args": vars(args), "best_epoch": ckpt["epoch"], "best_test": ckpt["test_metrics"],
                   "active_dims": active_dims, "n_params": n_params, "history": history}, f, indent=2)
    print(f"Figuras e métricas em {args.out_dir}/")


if __name__ == "__main__":
    main()


## 2. Configuração
- `LATENT_DIM` ($m$): dimensão do latente. 16 é um bom ponto de partida.
- `BETA`: peso de $L(x)$. **1.0 = fórmula exata do enunciado**; valores menores dão imagens mais nítidas.
- `FAKE = True` usa dados falsos só para testar o código rapidamente.

In [ ]:
LATENT_DIM = 16
BETA = 0.25
EPOCHS = 30
FAKE = False
FLAGS = f"--latent-dim {LATENT_DIM} --beta {BETA} --epochs {EPOCHS}" + (" --fake --num-workers 0" if FAKE else "")
print(FLAGS)

## 3. Conferir os dados

In [ ]:
import torch
from src.dataset import get_dataloaders, CLASS_NAMES
from src import vis_vae

train_loader, test_loader = get_dataloaders(batch_size=256, num_workers=0, fake=FAKE)
print('treino:', len(train_loader.dataset), '| teste:', len(test_loader.dataset))
x, y = next(iter(train_loader))
print('x:', tuple(x.shape), 'min/max:', float(x.min()), float(x.max()), '| y:', tuple(y.shape))
print('classes:', ''.join(CLASS_NAMES))
vis_vae.plot_dataset_grid(x, y)

## 4. Treinar o VAE
Salva o melhor checkpoint (menor loss de teste), os latentes e as figuras em `outputs/`.

In [ ]:
!python train_vae.py {FLAGS}

## 5. Resultados

In [ ]:
from IPython.display import Image, display
for name in ['vae_loss_curves', 'vae_reconstructions', 'vae_prior_samples', 'vae_interpolation', 'vae_latent_pca']:
    print(name)
    display(Image(f'outputs/{name}.png'))

**O que observar:**
- **Reconstruções** nítidas e com o caractere certo → o decoder está bom.
- **Amostras de $\mathcal{N}(0, I)$** (sem Flow) costumam sair borradas ou misturadas: é o *baseline* que o Flow precisa superar.
- **PCA** com classes agrupadas → o latente tem estrutura por classe (bom para o condicionamento/CFG).
- **Dimensões ativas** (no log do treino): se forem bem menos que $m$, o VAE está ignorando parte do latente (diminua `BETA`).

## 6. (Opcional) Comparar valores de β
Treinos curtos para escolher β. Útil para um slide de "decisões de engenharia".

In [ ]:
RUN_SWEEP = False
if RUN_SWEEP:
    import json
    for b in [1.0, 0.25, 0.05]:
        !python train_vae.py --latent-dim {LATENT_DIM} --beta {b} --epochs 10 --out-dir outputs/beta_{b} {'--fake --num-workers 0' if FAKE else ''} > /dev/null
        m = json.load(open(f'outputs/beta_{b}/metrics.json'))
        print(f"beta={b}: R={m['best_test']['recon_error']:.2f}  L={m['best_test']['latent_error']:.2f}  "
              f"dimensões ativas={m['active_dims']}/{LATENT_DIM}")
        display(Image(f'outputs/beta_{b}/vae_prior_samples.png'))

## 7. Como o resto do grupo usa a base
Exemplo para quem vai treinar o Flow ($\Phi$): carregar os latentes e o decoder.

In [ ]:
from src.vae import load_vae
device = 'cuda' if torch.cuda.is_available() else 'cpu'
vae = load_vae('outputs/vae.pt', device)
lat = torch.load('outputs/latents.pt')
mu, sigma, labels = lat['train']['mu'], lat['train']['sigma'], lat['train']['y']
print('mu:', tuple(mu.shape), '| sigma:', tuple(sigma.shape), '| labels:', tuple(labels.shape))

# alvo do Flow: z(1) = mu_E(x)  (ou z(1) = mu + sigma * W, amostrando do encoder)
z1 = mu[:8].to(device)
with torch.no_grad():
    imgs = vae.decode(z1)   # x = mu_D(z(1))
print('imagens decodificadas:', tuple(imgs.shape))

## 8. Salvar e compartilhar
Opção A: baixar um `.zip`. Opção B: copiar para o Google Drive e compartilhar a pasta com o grupo.

In [ ]:
!cd outputs && zip -q -r ../base_vae_outputs.zip vae.pt latents.pt metrics.json *.png
try:
    from google.colab import files
    files.download('base_vae_outputs.zip')
except ImportError:
    print('Fora do Colab: arquivo em base_vae_outputs.zip')

In [ ]:
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !mkdir -p /content/drive/MyDrive/ia-generativa-tp/base
    !cp outputs/vae.pt outputs/latents.pt outputs/metrics.json outputs/*.png /content/drive/MyDrive/ia-generativa-tp/base/